# 🧬 Session 2: OOP Advanced — Inheritance, Polymorphism & Design
### *Reuse, extend, and design with confidence*

---

| Info | Detail |
|------|--------|
| ⏱️ Duration | ~60 minutes |
| 📝 Exercises | 15 (with hints & solutions) |
| 🎯 Goal | Master inheritance, polymorphism, magic methods, dataclasses |
| 📋 Prerequisites | Session 1 (OOP Foundations) |

---

## 2.1 Inheritance — Building on Existing Classes 🧱

```python
class Parent:       # base / super class
    ...

class Child(Parent): # derived / sub class — inherits everything!
    ...
```

In [ ]:
class Animal:
    def __init__(self, name, species):
        self.name = name
        self.species = species
    
    def speak(self):
        return f"{self.name} makes a sound"
    
    def __str__(self):
        return f"{self.name} ({self.species})"

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name, "Dog")  # call parent's __init__
        self.breed = breed
    
    def speak(self):                   # override parent method
        return f"{self.name} says: Woof! 🐕"
    
    def fetch(self):                   # new method only for Dog
        return f"{self.name} fetches the ball! 🎾"

class Cat(Animal):
    def __init__(self, name, indoor=True):
        super().__init__(name, "Cat")
        self.indoor = indoor
    
    def speak(self):
        return f"{self.name} says: Meow! 🐱"

d = Dog("Buddy", "Golden Retriever")
c = Cat("Luna")

print(d)              # inherited __str__
print(d.speak())      # overridden
print(d.fetch())      # Dog-only
print(c.speak())      # overridden
print(f"Is Dog an Animal? {isinstance(d, Animal)}")  # True

### ✏️ Exercise 1 — Vehicle Hierarchy 🚗

In [ ]:
class Vehicle:
    def __init__(self, make, model, year):
        self.make, self.model, self.year = make, model, year
    def describe(self):
        return f"{self.year} {self.make} {self.model}"

# TODO: Create a Car class that inherits from Vehicle
# - adds 'doors' attribute (default 4)
# - overrides describe() to include doors
class Car(Vehicle):
    def __init__(self, make, model, year, doors=4):
        ___
        self.doors = doors
    def describe(self):
        return f"{super().describe()} ({self.doors}-door)"

# TODO: Create a Motorcycle that inherits from Vehicle
# - adds 'engine_cc' attribute
class Motorcycle(Vehicle):
    def __init__(self, make, model, year, cc):
        ___
        ___
    def describe(self):
        return f"{super().describe()} ({self.engine_cc}cc)"

c = Car("Tata", "Nexon", 2024)
m = Motorcycle("Royal Enfield", "Classic 350", 2024, 350)
print(c.describe())  # 2024 Tata Nexon (4-door)
print(m.describe())  # 2024 Royal Enfield Classic 350 (350cc)

*💡 Hint: `super().__init__(make, model, year)`. `self.engine_cc = cc`.*

### ✏️ Exercise 2 — Employee Types 👔

In [ ]:
class Employee:
    def __init__(self, name, base_salary):
        self.name = name
        self.base_salary = base_salary
    def total_pay(self):
        return self.base_salary
    def __str__(self):
        return f"{self.name}: ₹{self.total_pay():,}"

# TODO: Manager gets a 20% bonus
class Manager(Employee):
    def total_pay(self):
        return ___

# TODO: SalesPerson gets base + commission_rate * sales
class SalesPerson(Employee):
    def __init__(self, name, base_salary, commission_rate):
        super().__init__(name, base_salary)
        self.commission_rate = commission_rate
        self.sales = 0
    
    def make_sale(self, amount):
        self.sales += amount
    
    def total_pay(self):
        return ___

# TODO: Intern gets a fixed stipend only (ignore base_salary)
class Intern(Employee):
    def __init__(self, name, stipend):
        super().__init__(name, 0)
        self.stipend = stipend
    def total_pay(self):
        return ___

team = [
    Manager("Anita", 100000),
    SalesPerson("Rahul", 50000, 0.10),
    Intern("Sneha", 15000),
]
team[1].make_sale(200000)  # Rahul's sales

print("👥 Team Payroll:")
for emp in team:
    print(f"  {emp}")
# Anita: ₹120,000  Rahul: ₹70,000  Sneha: ₹15,000

*💡 Hint: Manager: `self.base_salary * 1.20`. Sales: `self.base_salary + self.commission_rate * self.sales`. Intern: `self.stipend`.*

---

## 2.2 Method Resolution Order (MRO) 🗺️

When a class inherits from multiple parents, Python uses **C3 linearization** to decide which method to call.

In [ ]:
class A:
    def greet(self): return "Hello from A"

class B(A):
    def greet(self): return "Hello from B"

class C(A):
    def greet(self): return "Hello from C"

class D(B, C):  # multiple inheritance!
    pass

d = D()
print(d.greet())       # B wins (left-to-right)
print(D.__mro__)       # D → B → C → A → object
print(D.mro())         # same, as a list

### ✏️ Exercise 3 — Predict the MRO 🔮

In [ ]:
class Base:
    def who(self): return "Base"

class Left(Base):
    def who(self): return "Left"

class Right(Base):
    pass  # inherits Base.who()

class Child(Left, Right):
    pass

class GrandChild(Child):
    pass

# Predict each, then run:
print(Child().who())       # Prediction: ___
print(Right().who())       # Prediction: ___
print(GrandChild().who())  # Prediction: ___
print([c.__name__ for c in GrandChild.__mro__])  # Prediction: ___

*💡 Hint: Child→Left, Right→Base, GrandChild→Left. MRO: GrandChild→Child→Left→Right→Base→object.*

---

## 2.3 Polymorphism — Same Interface, Different Behaviour 🎭

In [ ]:
# All shapes have area() and perimeter() — but compute them differently
import math

class Shape:
    def area(self): raise NotImplementedError
    def perimeter(self): raise NotImplementedError

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r**2
    def perimeter(self): return 2 * math.pi * self.r
    def __str__(self): return f"Circle(r={self.r})"

class Rect(Shape):
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h
    def perimeter(self): return 2 * (self.w + self.h)
    def __str__(self): return f"Rect({self.w}×{self.h})"

class Triangle(Shape):
    def __init__(self, a, b, c):
        self.a, self.b, self.c = a, b, c
    def perimeter(self): return self.a + self.b + self.c
    def area(self):  # Heron's formula
        s = self.perimeter() / 2
        return math.sqrt(s * (s-self.a) * (s-self.b) * (s-self.c))
    def __str__(self): return f"Triangle({self.a},{self.b},{self.c})"

# Polymorphism — same function works on ALL shapes
def print_shape_info(shape):
    print(f"  {shape}: area={shape.area():.2f}, perimeter={shape.perimeter():.2f}")

shapes = [Circle(5), Rect(4, 6), Triangle(3, 4, 5)]
for s in shapes:
    print_shape_info(s)  # same call, different results!

### ✏️ Exercise 4 — Polymorphic Payment System 💳

In [ ]:
class Payment:
    def __init__(self, amount):
        self.amount = amount
    def process(self):
        raise NotImplementedError

# TODO: CashPayment — just prints "Paid ₹X in cash"
class CashPayment(Payment):
    def process(self):
        ___

# TODO: CardPayment — adds 2% surcharge
class CardPayment(Payment):
    def __init__(self, amount, card_last4):
        super().__init__(amount)
        self.card_last4 = card_last4
    def process(self):
        total = ___
        return f"💳 Charged ₹{total:,.2f} to card ****{self.card_last4}"

# TODO: UPIPayment — no surcharge, shows UPI ID
class UPIPayment(Payment):
    def __init__(self, amount, upi_id):
        super().__init__(amount)
        self.upi_id = upi_id
    def process(self):
        return f"📱 Sent ₹{self.amount:,} to {self.upi_id}"

# Polymorphism in action
payments = [
    CashPayment(500),
    CardPayment(2000, "4321"),
    UPIPayment(1500, "priya@upi"),
]
for p in payments:
    print(p.process())

*💡 Hint: Cash: `return f"💵 Paid ₹{self.amount:,} in cash"`. Card total: `self.amount * 1.02`.*

---

## 2.4 Abstract Base Classes (ABC) 📋

An **abstract class** forces subclasses to implement certain methods.

In [ ]:
from abc import ABC, abstractmethod

class DatabaseConnector(ABC):
    @abstractmethod
    def connect(self): ...
    
    @abstractmethod
    def execute(self, query): ...
    
    def log(self, msg):  # concrete method — inherited as-is
        print(f"  📋 LOG: {msg}")

class MySQLConnector(DatabaseConnector):
    def connect(self):
        self.log("Connecting to MySQL...")
        return "Connected to MySQL"
    def execute(self, query):
        return f"MySQL executing: {query}"

# db = DatabaseConnector()  # ❌ TypeError: Can't instantiate abstract class!
db = MySQLConnector()
print(db.connect())
print(db.execute("SELECT * FROM users"))

### ✏️ Exercise 5 — Abstract Shape with ABC 📐

In [ ]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self): ...
    @abstractmethod
    def perimeter(self): ...
    
    def describe(self):  # concrete
        return f"{self.__class__.__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"

# TODO: Implement Square(side)
class Square(Shape):
    def __init__(self, side): self.side = side
    def area(self): return ___
    def perimeter(self): return ___

# TODO: Implement Ellipse(a, b)
# area = π × a × b
# perimeter ≈ π × (3(a+b) - √((3a+b)(a+3b)))
class Ellipse(Shape):
    def __init__(self, a, b): self.a, self.b = a, b
    def area(self): return ___
    def perimeter(self):
        return math.pi * (3*(self.a+self.b) - math.sqrt((3*self.a+self.b)*(self.a+3*self.b)))

shapes = [Square(5), Ellipse(4, 3)]
for s in shapes: print(s.describe())

# This should fail:
try:
    Shape()
except TypeError as e:
    print(f"\n❌ {e}")

*💡 Hint: Square: `self.side**2`, `4*self.side`. Ellipse area: `math.pi * self.a * self.b`.*

---

## 2.5 Magic / Dunder Methods 🪄

Make your objects behave like built-in types!

| Method | Operator | Example |
|--------|----------|---------|
| `__eq__` | `==` | `a == b` |
| `__lt__` | `<` | `a < b` (enables sorting!) |
| `__add__` | `+` | `a + b` |
| `__len__` | `len()` | `len(a)` |
| `__getitem__` | `[]` | `a[0]` |
| `__contains__` | `in` | `x in a` |

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y
    
    def __add__(self, other):       # v1 + v2
        return Vector(self.x + other.x, self.y + other.y)
    
    def __sub__(self, other):       # v1 - v2
        return Vector(self.x - other.x, self.y - other.y)
    
    def __mul__(self, scalar):      # v * 3
        return Vector(self.x * scalar, self.y * scalar)
    
    def __eq__(self, other):        # v1 == v2
        return self.x == other.x and self.y == other.y
    
    def __abs__(self):              # abs(v) → magnitude
        return (self.x**2 + self.y**2) ** 0.5
    
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

v1 = Vector(3, 4)
v2 = Vector(1, 2)
print(f"v1 + v2 = {v1 + v2}")
print(f"v1 * 3  = {v1 * 3}")
print(f"|v1|    = {abs(v1)}")
print(f"v1 == v2? {v1 == v2}")

### ✏️ Exercise 6 — `Money` Class with Operators 💰

In [ ]:
class Money:
    def __init__(self, amount, currency="INR"):
        self.amount = amount
        self.currency = currency
    
    def __add__(self, other):
        if self.currency != other.currency:
            raise ValueError("Can't add different currencies!")
        return Money(___)
    
    def __sub__(self, other):
        if self.currency != other.currency:
            raise ValueError("Can't subtract different currencies!")
        return Money(___)
    
    def __mul__(self, factor):
        return Money(___)
    
    def __eq__(self, other):
        return ___
    
    def __lt__(self, other):
        return ___
    
    def __repr__(self):
        return f"Money({self.amount}, '{self.currency}')"
    
    def __str__(self):
        symbols = {"INR": "₹", "USD": "$", "EUR": "€"}
        sym = symbols.get(self.currency, self.currency)
        return f"{sym}{self.amount:,.2f}"

a = Money(1000)
b = Money(500)
print(f"{a} + {b} = {a + b}")
print(f"{a} - {b} = {a - b}")
print(f"{a} × 3 = {a * 3}")
print(f"{a} == {b}? {a == b}")
print(f"{a} < {b}? {a < b}")
print(f"Sorted: {sorted([Money(300), Money(100), Money(200)])}")

*💡 Hint: `self.amount + other.amount, self.currency`. Mul: `self.amount * factor, self.currency`. Eq: `self.amount == other.amount and self.currency == other.currency`. Lt: `self.amount < other.amount`.*

### ✏️ Exercise 7 — `Deck` with `__len__`, `__getitem__`, `__contains__` 🃏

In [ ]:
import random

class Card:
    SUITS = ["♠", "♥", "♦", "♣"]
    RANKS = ["2","3","4","5","6","7","8","9","10","J","Q","K","A"]
    
    def __init__(self, rank, suit):
        self.rank, self.suit = rank, suit
    def __repr__(self): return f"{self.rank}{self.suit}"
    def __eq__(self, other): return self.rank == other.rank and self.suit == other.suit

class Deck:
    def __init__(self):
        self.cards = [Card(r, s) for s in Card.SUITS for r in Card.RANKS]
    
    def __len__(self):           # len(deck)
        return ___
    
    def __getitem__(self, idx):  # deck[0], deck[-1], deck[0:5]
        return ___
    
    def __contains__(self, card): # Card(...) in deck
        return ___
    
    def shuffle(self):
        random.shuffle(self.cards)
        return self
    
    def deal(self, n=1):
        dealt = self.cards[:n]
        self.cards = self.cards[n:]
        return dealt

deck = Deck()
print(f"Deck size: {len(deck)}")     # 52
print(f"First card: {deck[0]}")       # 2♠
print(f"Last 3: {deck[-3:]}")         # [Q♣, K♣, A♣]
print(f"A♠ in deck? {Card('A','♠') in deck}")  # True

deck.shuffle()
hand = deck.deal(5)
print(f"Hand: {hand}")
print(f"Remaining: {len(deck)}")

*💡 Hint: `len(self.cards)`, `self.cards[idx]`, `card in self.cards`.*

### ✏️ Exercise 8 — `__iter__` — Make Objects Iterable 🔄

In [ ]:
class NumberRange:
    """A custom range that supports iteration."""
    def __init__(self, start, end):
        self.start, self.end = start, end
    
    def __iter__(self):
        self._current = self.start
        return self
    
    def __next__(self):
        if self._current > self.end:
            raise StopIteration
        val = self._current
        self._current += 1
        return val
    
    def __len__(self):
        return max(0, self.end - self.start + 1)
    
    def __contains__(self, val):
        return self.start <= val <= self.end

r = NumberRange(1, 5)
print(list(r))             # [1, 2, 3, 4, 5]
print(f"Length: {len(r)}")  # 5
print(f"3 in range? {3 in r}")  # True
print(f"Sum: {sum(r)}")

### ✏️ Exercise 9 — Make a `StudentGradebook` Sortable 📊

In [ ]:
class GradeStudent:
    def __init__(self, name, gpa):
        self.name, self.gpa = name, gpa
    
    # TODO: Enable sorting by GPA
    def __lt__(self, other): return ___
    def __eq__(self, other): return ___
    def __repr__(self): return f"{self.name}({self.gpa})"

students = [
    GradeStudent("Anita", 9.2),
    GradeStudent("Rahul", 7.5),
    GradeStudent("Sneha", 9.8),
    GradeStudent("Vishal", 8.1),
]

print(f"Sorted: {sorted(students)}")
print(f"Top: {max(students)}")
print(f"Min: {min(students)}")

*💡 Hint: `self.gpa < other.gpa`, `self.gpa == other.gpa`.*

---

## 2.6 `dataclasses` — Auto-Generated OOP 🤖

Python 3.7+ can **auto-generate** `__init__`, `__repr__`, `__eq__`, and more!

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Point:
    x: float
    y: float
    label: str = ""  # default value

p1 = Point(3, 4, "A")
p2 = Point(3, 4, "A")
p3 = Point(1, 2)

print(p1)            # Point(x=3, y=4, label='A') — auto __repr__!
print(p1 == p2)      # True — auto __eq__!
print(p1 == p3)      # False

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass(order=True)   # enables <, >, sorting!
class Student:
    sort_index: float = field(init=False, repr=False)  # hidden sort key
    name: str
    gpa: float
    courses: List[str] = field(default_factory=list)
    
    def __post_init__(self):
        self.sort_index = self.gpa  # sort by GPA

students = [
    Student("Anita", 9.2, ["Math", "Physics"]),
    Student("Rahul", 7.5),
    Student("Sneha", 9.8, ["CS", "Math", "English"]),
]

for s in sorted(students, reverse=True):
    print(f"  {s}")

### ✏️ Exercise 10 — `@dataclass` Practice 🤖

In [ ]:
from dataclasses import dataclass, field
from typing import List

# TODO: Create a Product dataclass
@dataclass
class Product:
    name: str
    price: float
    category: str = "General"
    tags: List[str] = field(default_factory=list)
    
    @property
    def price_with_gst(self):
        return self.price * 1.18

p1 = Product("Laptop", 75000, "Electronics", ["tech", "portable"])
p2 = Product("Laptop", 75000, "Electronics", ["tech", "portable"])
p3 = Product("Mouse", 500)

print(p1)
print(f"p1 == p2? {p1 == p2}")  # True
print(f"GST price: ₹{p1.price_with_gst:,.2f}")
print(f"Mouse: {p3}")

### ✏️ Exercise 11 — Inheritance with Dataclasses 🧬

In [ ]:
from dataclasses import dataclass

@dataclass
class Person:
    name: str
    age: int

# TODO: Create Employee dataclass inheriting from Person
@dataclass
class DCEmployee(Person):
    department: str = "General"
    salary: float = 50000
    
    @property
    def annual(self):
        return self.salary * 12

e = DCEmployee("Anita", 28, "Engineering", 95000)
print(e)
print(f"Annual: ₹{e.annual:,}")
print(f"Is Person? {isinstance(e, Person)}")

### ✏️ Exercise 12 — `__enter__` / `__exit__` — Context Manager 🚪

In [ ]:
# Make an object usable with 'with' statement

class Timer:
    """Context manager that times code blocks."""
    import time
    
    def __init__(self, label="Timer"):
        self.label = label
    
    def __enter__(self):
        import time
        self.start = time.time()
        print(f"⏱️ {self.label} started...")
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        import time
        self.elapsed = time.time() - self.start
        print(f"⏱️ {self.label} finished in {self.elapsed:.4f}s")
        return False  # don't suppress exceptions

with Timer("Sum calculation"):
    total = sum(range(1_000_000))
    print(f"   Sum = {total:,}")

### ✏️ Exercise 13 — Predict the Output: Inheritance 🔮

In [ ]:
class A:
    def __init__(self): self.x = 1
    def show(self): print(f"A.x = {self.x}")

class B(A):
    def __init__(self):
        super().__init__()
        self.y = 2

class C(B):
    def __init__(self):
        super().__init__()
        self.x = 99  # overrides A's x!

# Predict each, then run:
a = A(); a.show()        # Prediction: ___
b = B(); b.show()        # Prediction: ___
c = C(); c.show()        # Prediction: ___
print(hasattr(b, 'y'))   # Prediction: ___
print(isinstance(c, A))  # Prediction: ___

*💡 Hint: A.x=1, A.x=1, A.x=99 (overridden!), True, True.*

### ✏️ Exercise 14 — Build a Custom List ✨

In [ ]:
class SortedList:
    """A list that stays sorted at all times."""
    def __init__(self, items=None):
        self._data = sorted(items or [])
    
    def add(self, item):
        import bisect
        bisect.insort(self._data, item)
    
    def __len__(self): return len(self._data)
    def __getitem__(self, idx): return self._data[idx]
    def __contains__(self, item): return item in self._data
    def __repr__(self): return f"SortedList({self._data})"
    def __iter__(self): return iter(self._data)
    
    def __add__(self, other):
        """Merge two sorted lists."""
        return SortedList(self._data + other._data)

sl = SortedList([5, 2, 8, 1])
print(sl)          # SortedList([1, 2, 5, 8])
sl.add(3)
sl.add(6)
print(sl)          # SortedList([1, 2, 3, 5, 6, 8])
print(f"len: {len(sl)}")
print(f"sl[0]: {sl[0]}, sl[-1]: {sl[-1]}")
print(f"3 in sl? {3 in sl}")
print(f"Sum: {sum(sl)}")

sl2 = SortedList([4, 7])
print(f"Merged: {sl + sl2}")

---
## 🎯 Exercise 15 — Mini-Project: Shape Hierarchy 📐

In [ ]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self): ...
    @abstractmethod
    def perimeter(self): ...
    def __lt__(self, other): return self.area() < other.area()
    def __eq__(self, other): return abs(self.area() - other.area()) < 0.001
    def __repr__(self): return f"{self.__class__.__name__}(area={self.area():.2f})"

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r**2
    def perimeter(self): return 2 * math.pi * self.r

class Rectangle(Shape):
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h
    def perimeter(self): return 2 * (self.w + self.h)

class Square(Rectangle):
    def __init__(self, s): super().__init__(s, s)

class RightTriangle(Shape):
    def __init__(self, base, height):
        self.base, self.height = base, height
    def area(self): return 0.5 * self.base * self.height
    def perimeter(self):
        hyp = math.sqrt(self.base**2 + self.height**2)
        return self.base + self.height + hyp

# Create shapes and compare
shapes = [
    Circle(5),
    Rectangle(4, 8),
    Square(6),
    RightTriangle(6, 8),
]

print("📐 SHAPE COMPARISON")
print("═" * 50)
for s in shapes:
    print(f"  {s.__class__.__name__:<15} area={s.area():>8.2f}  perimeter={s.perimeter():>8.2f}")

print(f"\n🏆 Largest:  {max(shapes)}")
print(f"🔻 Smallest: {min(shapes)}")
print(f"📊 Sorted:   {sorted(shapes)}")

total = sum(s.area() for s in shapes)
print(f"📏 Total area: {total:.2f}")

---
## 🎊 Session 2 Complete!

| ✅ | Concept | Key Takeaway |
|---|---------|-------------|
| 🧱 | Inheritance | `class Child(Parent)`, `super()` |
| 🗺️ | MRO | Left-to-right, depth-first resolution |
| 🎭 | Polymorphism | Same method, different behavior |
| 📋 | ABC | `@abstractmethod` forces implementation |
| 🪄 | Magic methods | `__add__`, `__eq__`, `__lt__`, `__len__`, `__getitem__` |
| 🤖 | dataclasses | Auto `__init__`, `__repr__`, `__eq__` |
| 🚪 | Context managers | `__enter__`/`__exit__`, `with` |

### 🚀 Next: Session 3 — Iterators, Generators & Collections
---

## 📖 Solutions

<details>
<summary><strong>Click to reveal all solutions</strong></summary>

### Ex 1: `super().__init__(make, model, year)`. `self.engine_cc = cc`.
### Ex 2: `self.base_salary * 1.20`. `self.base_salary + self.commission_rate * self.sales`. `self.stipend`.
### Ex 3: Left, Base, Left. MRO: GrandChild→Child→Left→Right→Base→object.
### Ex 4: `return f"💵 Paid ₹{self.amount:,} in cash"`. `self.amount * 1.02`.
### Ex 5: `self.side**2`, `4*self.side`. `math.pi * self.a * self.b`.
### Ex 6: `self.amount+other.amount, self.currency`. `self.amount*factor, self.currency`. `self.amount==other.amount and self.currency==other.currency`. `self.amount<other.amount`.
### Ex 7: `len(self.cards)`, `self.cards[idx]`, `card in self.cards`.
### Ex 9: `self.gpa < other.gpa`, `self.gpa == other.gpa`.
### Ex 13: 1, 1, 99, True, True.

</details>